Lấy 3 chỉ số chứng khoán, so sánh mức độ tăng giảm so với ngày hôm trước

In [ ]:
from vnstock import Quote
import pandas as pd
from datetime import timedelta,time

symbols = ['VNINDEX', 'VN30', 'HNX30']
data_frames = []

for sym in symbols:
    quote = Quote(symbol=sym, source='VCI')
    df = quote.history(length='1', interval='d').reset_index()  # reset index để có cột time thay vì index
    # chèn cột mới ở vị trí 0 (đầu DataFrame)
    df.insert(0, 'symbol', sym)
    # df = df.df[['symbol', 'time', 'close', 'volume']]
    # trong df, lấy ra các cột cần thiết: symbol, time, close, volume
    df = df[['symbol', 'time', 'close', 'volume']]
    data_frames.append(df)
    
all_data = pd.concat(data_frames, ignore_index=True)

# trong từng symsbol, lấy giá của ngày d-1, gán vào ngày d-2, tính chênh lệch điểm và phần trăm thay đổi
all_data['prev_close'] = all_data.groupby('symbol')['close'].shift(1)
all_data['change'] = all_data['close'] - all_data['prev_close']
all_data['change_pct'] = (all_data['change'] / all_data['prev_close'] * 100).round(2)	

today = pd.Timestamp.now().date().strftime('%Y-%m-%d')
all_data_today = all_data[all_data['time'] == today]

all_data_today.reset_index()
# all_data.to_csv('index_data.csv', index=False)

,index,symbol,time,close,volume,prev_close,change,change_pct
0,2,VNINDEX,2026-03-13,1696.24,1023068442,1709.61,-13.37,-0.78
1,5,VN30,2026-03-13,1853.60,372106248,1859.80,-6.20,-0.33
2,8,HNX30,2026-03-13,524.78,94164714,532.70,-7.92,-1.49


Non-200 response fetching content: 400


In [4]:
all_data_today.to_csv('index_data.csv', index=False)

# 2. lấy top 10 cổ phiếu vốn hóa giảm

In [ ]:
# code lấy top 10 cổ phiếu có vốn hóa giảm/tăng
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
vn30 = pd.read_csv('ticker_vn30.csv')           
vn30 = vn30['symbol'].astype(str) + ".VN"  
# thêm ".VN" vào mỗi mã để phù hợp với yfinance


start=(datetime.today() - timedelta(days=2)).strftime("%Y-%m-%d")
end= datetime.today().strftime("%Y-%m-%d")
# trong yfinance,lấy giá đóng cửa của ngày start và end
# từ đó tính ra thay đổi điểm và phần trăm thay đổi của cổ phiếu trong khoảng thời gian đó

records = []
for ticker in vn30:
	data = yf.download(ticker, start=start, end=end)
	sharesOutstanding = yf.Ticker(ticker).info.get('sharesOutstanding', 'N/A')
	if data.empty:
		print(f"No data for {ticker} between {start} and {end}.")
		continue
	close_start = data['Close'].iloc[0]
	close_end = data['Close'].iloc[-1]
	sharesOutstanding = sharesOutstanding
	change_points = close_end - close_start
	change_percent = (change_points / close_start)*100
	market_cap_d_2ays_ago = (close_start * sharesOutstanding)/1e9
	maket_cap_now = (close_end * sharesOutstanding)/1e9
	change_market_cap_bil_vnd = maket_cap_now - market_cap_d_2ays_ago
	records.append({
		'Ticker': ticker.replace(".VN", "")
		,'Close End': close_end
		,'Change Points': change_points
		,'Change Percent': change_percent
		,'Market Cap 2 Days Ago (Bil VND)': market_cap_d_2ays_ago
		,'Market Cap Now (Bil VND)': maket_cap_now
		,'Change Market Cap (Bil VND)': change_market_cap_bil_vnd
		})

records= pd.DataFrame(records)
# Chuyển các cột về kiểu float để chỉ hiển thị số
records['Close End'] = records['Close End'].astype(float)
records['Change Points'] = records['Change Points'].astype(float)
records['Change Percent'] = records['Change Percent'].astype(float)
records['Market Cap 2 Days Ago (Bil VND)'] = records['Market Cap 2 Days Ago (Bil VND)'].astype(float)
records['Market Cap Now (Bil VND)'] = records['Market Cap Now (Bil VND)'].astype(float)
records['Change Market Cap (Bil VND)'] = records['Change Market Cap (Bil VND)'].astype(float)

records.sort_values(by='Change Market Cap (Bil VND)', ascending=False, inplace=True)
records.reset_index(drop=True, inplace=True)
records

In [27]:
records.to_csv('top_10_high.csv')

# 3. get stock high performance

In [ ]:
df = pd.read_csv('ticker_hose.csv')
df = df['symbol'].astype(str) + ".VN"

In [ ]:
scanner = []
for ticker in df:
    info = yf.Ticker(ticker).info
    price = info.get('regularMarketPrice', '')
    ROE = info.get('returnOnEquity', '')*100
    ROA = info.get('returnOnAssets', '')*100
    ROIC = info.get('returnOnInvestedCapital', '')*100
    PE = info.get('trailingPE', '')
    PB = info.get('priceToBook','')
    EPS = info.get('trailingEps','')
    
    scanner.append({
    'Ticker': ticker.replace(".VN", "")
    ,'Price': price
    ,'ROE': ROE
    ,'ROA': ROA
    ,'ROIC':ROIC
    ,'PE': PE
    ,'PB': PB
    ,'EPS': EPS

  })
scanner_df = pd.DataFrame(scanner)

for col in ['ROE','ROA','ROIC','PE','PB','EPS']:
    scanner_df[col] = pd.to_numeric(scanner_df[col], errors='coerce')


In [ ]:
# Tạo cột xếp hạng roe, pe, pb
scanner_df['ROE_Rank'] = scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom')
scanner_df['PE_Rank'] = scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')
scanner_df['ROE + PE'] = (scanner_df['ROE'].rank(method='dense',ascending=False, na_option='bottom') 
                          + 
                          (scanner_df['PE'].rank(method='dense',ascending=True, na_option='bottom')))
scanner_df['ROE + PE Ranking'] = scanner_df['ROE + PE'].rank(method='dense',ascending=True, na_option='bottom').astype(int)
scanner_df.sort_values(by='ROE + PE Ranking',ascending=True).reset_index()

In [54]:
scanner_df.to_csv('ranking_ticker.csv')

# 4. Lấy 3 bảng: income_statement, balance_sheet, valuation_measures

In [ ]:
def get_income_statement(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    stmt = yf.Ticker(ticker).get_income_stmt(freq='quarterly')
    df = pd.DataFrame(stmt).loc[['TotalRevenue', 'CostOfRevenue', 'GrossProfit', 'NetIncomeContinuousOperations']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'TotalRevenue': df.at['TotalRevenue', date_col] / 1e9,
            'CostOfRevenue': df.at['CostOfRevenue', date_col] / 1e9,
            'GrossProfit': df.at['GrossProfit', date_col] / 1e9,
            'NetIncome': df.at['NetIncomeContinuousOperations', date_col] / 1e9,
        }
        row['GrossProfitMargin'] = 100.0*round(row['GrossProfit'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        row['NetIncomeMargin'] = 100.0*round(row['NetIncome'] / row['TotalRevenue'], 2) if row['TotalRevenue'] else None
        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_income_statements(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_income_statement(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_stmt = get_income_statements(tickers)
merged_stmt

In [ ]:
# 1 số stock không share dữ liệu lên yfinance
merged_stmt = merged_stmt.dropna(subset=['TotalRevenue','CostOfRevenue','GrossProfit','NetIncome','GrossProfitMargin','NetIncomeMargin'])


In [57]:
merged_stmt.to_csv('income_statement.csv')

In [ ]:
def get_balance_sheet(ticker):
    """Get income statement for a single ticker, return as a flat DataFrame with Ticker and Date columns."""
    bls = yf.Ticker(ticker).get_balance_sheet(freq='quarterly')
    df = pd.DataFrame(bls).loc[['TotalAssets','TotalLiabilitiesNetMinorityInterest','TotalEquityGrossMinorityInterest']]
    
    records = []
    for date_col in df.columns:
        row = {
            'Ticker': ticker.replace(".VN",""),
            'Date': date_col.strftime('%Y-%m-%d'),
            'Total Assets': df.at['TotalAssets', date_col] / 1e9,
            'Total Liabilities': df.at['TotalLiabilitiesNetMinorityInterest', date_col] / 1e9,
            'Total Equity': df.at['TotalEquityGrossMinorityInterest', date_col] / 1e9,
        }
        row['Liabilities/Equity ratio'] = round(row['Total Liabilities'] / row['Total Equity'], 2) if row['Total Equity'] else None

        records.append(row)
    
    result = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)
    return result

def get_balance_sheets(tickers):
    """Get income statements for multiple tickers and merge into one DataFrame."""
    all_data = []
    for ticker in tickers:
        try:
            df = get_balance_sheet(ticker)
            all_data.append(df)
            print(f"OK: {ticker}")
        except Exception as e:
            print(f"SKIP: {ticker} - {e}")
    return pd.concat(all_data, ignore_index=True) if all_data else pd.DataFrame()

tickers = df
merged_balance_sheet = get_balance_sheets(tickers)
merged_balance_sheet.dropna(subset= ['Total Assets','Total Liabilities','Total Equity','Liabilities/Equity ratio'])

In [59]:
merged_balance_sheet.to_csv('balance_sheet.csv')

# 5. lấy giá và sma20

In [ ]:
import yfinance as yf
import pandas as pd
import talib
from datetime import datetime,timedelta


start = ((datetime.today() - timedelta(days=120)).replace(day=1).strftime("%Y-%m-%d"))
end = datetime.today().strftime("%Y-%m-%d")

def get_price(ticker):
    """Get price history for a single ticker, return DataFrame with Ticker and Date columns."""
    hist = yf.download(ticker, start=start, end=end, interval='1d')
    close = hist['Close'].reset_index()
    df = close.melt(var_name='Ticker', id_vars='Date', value_name='Close')
    df = df.sort_values(by=['Ticker', 'Date'])
    df['Ticker'] = df['Ticker'].str.replace('.VN', '')
    df['SMA20'] = talib.SMA(df['Close'], timeperiod=20)
    return df

def get_prices(tickers):
    all_hist = []
    for ticker in tickers:
        get_hist = get_price(ticker)
        all_hist.append(get_hist)
    result = pd.concat(all_hist).sort_values(by=['Ticker','Date']).reset_index(drop=True)
    return result

merged = get_prices(df)
merged

In [61]:
merged.to_csv('hist_price_data.csv')